# 00 — Fantasy ETL Flow (manual/scheduled orchestrator front-end)

Thin front-end over `scripts/run_pipeline.py` — toggle which **groups**
(domain areas) to run, then this notebook builds a `--group` arg and calls
`run_pipeline.main()` directly (import + call, never a second subprocess
layer), so this notebook and the scheduled headless path
(`run_weekly.ps1` -> `run_pipeline.py`) are always the same code.

Groups are orthogonal to `run_pipeline.py`'s existing `phase` concept
(calendar timing: INSEASON/PRESEASON/OFFSEASON) — a group is *what* runs,
a phase is *when* it's allowed to run. Both filters combine with AND.

- **pre_season** — dim seeds (`01f`, `01e`) + dynasty rankings prep (`04b`,
  + `04c`/`04y` under `--profile dynasty`).
- **rookie** — auto-safe rookie-ranking scrapes only (`03a`-`03d`). Stops
  before `03x` (manual Excel) and `03y`/`03z` (fuzzy-review + apply) — those
  need a human in the loop and are never run from here. This cell prints a
  reminder to run them by hand.
- **regular_season** — the weekly in-season refresh chain (`04a` scrape ->
  `04z` crosswalk -> `04v` minors -> `02d` ledger -> `02e` derive).
- **injury** — **placeholder, currently a no-op.** No
  `fact_nfl_season_injuries` notebook exists yet (still a documented future
  gap in `.claude/memory/data-model.md`); toggling this on today runs
  nothing and prints a message. This is intentional scaffolding for when
  that source gets built, not a bug.

In [1]:
# --- toggles: flip these, then run all cells below ---
RUN_PRESEASON = False
RUN_ROOKIE = False
RUN_REGULAR_SEASON = True
RUN_INJURY = False

# Passed through to run_pipeline.main() as sys.argv; None = let run_pipeline
# derive the phase from today's date + the season calendar (same as the
# scheduled run).
PHASE_OVERRIDE = None   # e.g. "OFFSEASON" to force a phase for testing
DRY_RUN = True          # flip to False for a real run
NO_COMMIT = True        # scheduled run commits data/*.parquet on main only;
                        # keep True here unless you intend that side effect
PROFILE = None          # "dynasty" to append 04b -> 04c -> 04y under pre_season

In [2]:
import sys
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "scripts"))
import run_pipeline

In [3]:
groups = []
if RUN_PRESEASON:
    groups.append("pre_season")
if RUN_ROOKIE:
    groups.append("rookie")
if RUN_REGULAR_SEASON:
    groups.append("regular_season")
if RUN_INJURY:
    print("[info] RUN_INJURY=True: no injury notebooks registered yet "
          "(fact_nfl_season_injuries is still an unbuilt future source) "
          "— this toggle is a documented no-op placeholder.")

if RUN_ROOKIE:
    print("[reminder] rookie group runs 03a-03d only. After this completes, "
          "run 03y (dim_player_alias backfill) and 03z (apply fuzzy review) "
          "by hand — they need human review and are never run from here.")

if not groups:
    print("[info] no groups selected (RUN_INJURY-only counts as none) — "
          "nothing to run.")

print(f"[info] groups={groups or '(none)'} phase_override={PHASE_OVERRIDE} "
      f"dry_run={DRY_RUN} no_commit={NO_COMMIT} profile={PROFILE}")

[info] groups=['regular_season'] phase_override=None dry_run=True no_commit=True profile=None


In [4]:
argv = ["run_pipeline.py"]
if PHASE_OVERRIDE:
    argv += ["--phase", PHASE_OVERRIDE]
if groups:
    argv += ["--group", ",".join(groups)]
if PROFILE:
    argv += ["--profile", PROFILE]
if DRY_RUN:
    argv += ["--dry-run"]
if NO_COMMIT:
    argv += ["--no-commit"]

if groups:
    sys.argv = argv
    exit_code = run_pipeline.main()
    print(f"\n[info] run_pipeline.main() exit code: {exit_code}")

[info] phase=OFFSEASON week=PRE [DRY RUN]
[step] 04a_scrape: C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\notebooks\04a_fantrax_weekly_scrape.py
[step] 04z_crosswalk: -m nbconvert --to notebook --execute --inplace C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\notebooks\04z_fantrax_crosswalk.ipynb
[step] 04v_minor_contracts: C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\notebooks\04v_minor_contracts.py
[step] 02d_ledger: C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\notebooks\02d_fact_roster_transactions.py
[step] 02e_derive: C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\notebooks\02e_fact_fantasy_teams_derive.py
[info] commit skipped (flag)

**Pipeline 2026-07-18** — phase OFFSEASON, week PRE
▫️ 04a_scrape
▫️ 04z_crosswalk
▫️ 04v_minor_contracts
▫️ 02d_ledger
▫️ 02e_derive
**Review queues:**
• review_fuzzy_matches.csv: 4 r